# CodeBERT paired seed replication on Kaggle

Use **GPU T4 x2**, **Internet on**, and the same pinned environment as Versions 2–3.
Save Version → Save & Run All. This runs seeds **7 and 13** across control, module-only,
newline-only and combined module/newline datasets. Comments and all source CSVs stay unchanged.

Each seed controls both splitting and training. Actual source-row assignments must match
across all four datasets before training starts. Models: CodeBERT LoRA (FP32), LinearSVC,
and balanced LinearSVC. Hyperparameters match the seed-42 runs.

Download `results.zip` from Output after completion. Its timestamped folder contains
`seed_<seed>/<dataset>/{results,logs,data,split_manifest.json}`, runtime metadata and package versions.
Import with `python scripts/collect_seed_results.py /path/to/results.zip` to update the organized study.


In [ ]:
import shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/alexsmyr0/thesis-experiments"
REPO = Path("/tmp/thesis-experiments")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)], check=True)
# Use the notebook's interpreter and install every pipeline dependency.
# Stop immediately if setup fails. Existing compatible packages (including CUDA torch) stay installed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
# Kaggle may preinstall an old optional quantization backend that breaks PEFT adapter creation.
# CodeBERT uses unquantized weights, so remove that unused backend in this session.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Source commit:", COMMIT)


In [ ]:
# Check in a fresh process, matching the interpreter used by the training scripts.
# This avoids stale imports if Kaggle loaded a package before pip updated it.
import json

probe = subprocess.check_output([sys.executable, "-c", """
import importlib.metadata as metadata
import json
import torch
import transformers, peft, datasets, accelerate, sklearn, pandas, matplotlib, seaborn
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in Session options, then restart"
from transformers import TrainingArguments
assert TrainingArguments(output_dir='/tmp/kaggle-preflight', warmup_steps=0.1, report_to='none').get_warmup_steps(100) == 10
# Exercise adapter creation and a GPU backward pass before starting expensive jobs.
from transformers import RobertaConfig, RobertaForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model
model = get_peft_model(RobertaForSequenceClassification(RobertaConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=1,
    num_attention_heads=4, intermediate_size=64, num_labels=5)),
    LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
               lora_dropout=0.05, target_modules=['query', 'value'], bias='none')).cuda()
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
loss = model(input_ids=torch.full((4, 16), 2, device='cuda'),
             attention_mask=torch.ones((4, 16), dtype=torch.long, device='cuda'),
             labels=torch.tensor([0, 1, 2, 3], device='cuda')).loss.mean()
assert torch.isfinite(loss), 'LoRA GPU smoke test produced a non-finite loss'
loss.backward()
torch.cuda.synchronize()
del model, loss
torch.cuda.empty_cache()
packages = ['torch', 'transformers', 'peft', 'datasets', 'accelerate',
            'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']
print(json.dumps({
    'versions': {name: metadata.version(name) for name in packages},
    'cuda': torch.version.cuda,
    'lora_gpu_smoke_test': 'passed',
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}))
"""], text=True)
RUNTIME = json.loads(probe)
print(json.dumps(RUNTIME, indent=2))


In [ ]:
import os
from datetime import datetime, timezone

SEEDS = [7, 13]
DATASETS = ["control_unmodified", "only_modules", "only_newlines", "modules_newlines"]
CODEBERT = "lora"
TFIDF = "svm"
FP16 = False
RUN_NAME = datetime.now(timezone.utc).strftime("kaggle_%Y%m%dT%H%M%S_%fZ")
OUT = Path("/kaggle/working/experiments") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=False)
(OUT / "run_metadata.json").write_text(json.dumps({
    'run': RUN_NAME, 'repo': REPO_URL, 'commit': COMMIT,
    'seeds': SEEDS, 'split_seed_equals_training_seed': True,
    'datasets': DATASETS, 'codebert': CODEBERT, 'tfidf': TFIDF, 'fp16': FP16, **RUNTIME,
}, indent=2))
(OUT / "pip_freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
subprocess.run([sys.executable, str(REPO / "scripts/run_seed_sweep.py"),
                "--seeds", *map(str, SEEDS), "--out", str(OUT)], check=True)


In [ ]:
for seed in SEEDS:
    for name in DATASETS:
        print(f"\n=== seed={seed} dataset={name} ===")
        print((OUT / f"seed_{seed}" / name / "logs/summary.txt").read_text())
shutil.make_archive("/kaggle/working/results", "zip", OUT.parent, OUT.name)
print(f"Wrote /kaggle/working/results.zip (contains {RUN_NAME}/)")
